This notebook measures boat buying demand coming from the Greater Miami metro over the last twelve months. Greater Miami means the buyer location country is USA, the buyer location region is Florida, and the buyer location city is one of the tri county cities (Miami Dade, Broward, Palm Beach) listed in every query below. The scope is demand from Miami buyers. A usable lead is non spam, meaning IS_SPAM_FINAL equal to zero.

This first query was run before building the metro list, to see which Florida buyer cities actually appear in the data.

In [ ]:
SELECT
  USER_LOCATION_CITY AS city,
  COUNT(*)           AS leads
FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
  AND COALESCE(IS_SPAM_FINAL,0)=0
  AND USER_LOCATION_COUNTRY = 'USA'
  AND USER_LOCATION_REGION = 'Florida'
  AND NULLIF(TRIM(USER_LOCATION_CITY),'') IS NOT NULL
GROUP BY 1
ORDER BY leads DESC
LIMIT 45;

This is the leads level of demand. It reports how many leads Miami buyers sent, how many distinct buyers that is, what share of all United States demand this represents, how many of those leads are on dealer boats, and the dealer conversion rate, where a conversion is a boat that sold on or after the lead day. The conversion rate reads directly because the average of a one or zero flag equals the share that are one.

In [ ]:
WITH metro AS (
  SELECT *
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND COALESCE(IS_SPAM_FINAL,0)=0
    AND USER_LOCATION_COUNTRY='USA'
    AND USER_LOCATION_REGION='Florida'
    AND USER_LOCATION_CITY IN (
      'Miami','Miami Beach','Hialeah','Kendall','Doral','Homestead','Aventura','Coral Gables','North Miami','North Miami Beach','Cutler Bay','Miami Gardens','Pinecrest','Palmetto Bay','Sunny Isles Beach','Key Biscayne','Miami Lakes','Hialeah Gardens','Opa-locka',
      'Fort Lauderdale','Hollywood','Pompano Beach','Deerfield Beach','Coral Springs','Pembroke Pines','Miramar','Davie','Plantation','Sunrise','Hallandale Beach','Weston','Tamarac','Margate','Coconut Creek','Lauderhill','Dania Beach','Oakland Park','Wilton Manors','Parkland','Cooper City',
      'West Palm Beach','Boca Raton','Delray Beach','Boynton Beach','Palm Beach Gardens','Jupiter','Wellington','Palm Beach','Lake Worth','Royal Palm Beach','Riviera Beach','Greenacres','Lake Park','North Palm Beach')
),
sold AS (
  SELECT LISTING_ID, MIN(DATE_SOLD) AS fs
  FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS
  GROUP BY 1
),
us_all AS (
  SELECT COUNT(*) AS us_leads
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND COALESCE(IS_SPAM_FINAL,0)=0
    AND USER_LOCATION_COUNTRY='USA'
)
SELECT
  COUNT(*)                              AS metro_leads,
  COUNT(DISTINCT SENDER_EMAIL)          AS distinct_buyers,
  (SELECT us_leads FROM us_all)         AS all_us_leads,
  ROUND(COUNT(*) * 100.0 / (SELECT us_leads FROM us_all), 1) AS pct_of_us_demand,
  COUNT_IF(NOT IS_FSBO)                 AS dealer_leads,
  ROUND(AVG(CASE WHEN NOT IS_FSBO
                 THEN IFF(s.fs IS NOT NULL AND s.fs >= metro.DAY, 1, 0)
            END) * 100, 1)              AS dealer_conv_rate
FROM metro
LEFT JOIN sold s ON s.LISTING_ID = metro.LISTING_ID;

This is the monthly trend of Miami demand, used to judge seasonality. Miami runs flatter than the country, with a winter trough that stays close to half of the spring peak, consistent with warm weather boating year round.

In [ ]:
WITH metro AS (
  SELECT DAY
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND COALESCE(IS_SPAM_FINAL,0)=0
    AND USER_LOCATION_COUNTRY='USA'
    AND USER_LOCATION_REGION='Florida'
    AND USER_LOCATION_CITY IN (
      'Miami','Miami Beach','Hialeah','Kendall','Doral','Homestead','Aventura','Coral Gables','North Miami','North Miami Beach','Cutler Bay','Miami Gardens','Pinecrest','Palmetto Bay','Sunny Isles Beach','Key Biscayne','Miami Lakes','Hialeah Gardens','Opa-locka',
      'Fort Lauderdale','Hollywood','Pompano Beach','Deerfield Beach','Coral Springs','Pembroke Pines','Miramar','Davie','Plantation','Sunrise','Hallandale Beach','Weston','Tamarac','Margate','Coconut Creek','Lauderhill','Dania Beach','Oakland Park','Wilton Manors','Parkland','Cooper City',
      'West Palm Beach','Boca Raton','Delray Beach','Boynton Beach','Palm Beach Gardens','Jupiter','Wellington','Palm Beach','Lake Worth','Royal Palm Beach','Riviera Beach','Greenacres','Lake Park','North Palm Beach')
)
SELECT
  TO_CHAR(DATE_TRUNC('month', DAY), 'YYYY-MM') AS month,
  COUNT(*)                                     AS metro_leads
FROM metro
GROUP BY 1
ORDER BY 1;

This is the listings level, showing what boats Miami buyers demand. It reports the top boat classes by lead count, each class share of metro leads, and the median asking price for that class. Center console powerboats dominate, which fits the South Florida market.

In [ ]:
WITH metro AS (
  SELECT PRODUCT_CLASS, PRODUCT_PRICE, LISTING_STATE, USER_LISTING_DISTANCE_IN_MILES
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND COALESCE(IS_SPAM_FINAL,0)=0
    AND USER_LOCATION_COUNTRY='USA'
    AND USER_LOCATION_REGION='Florida'
    AND USER_LOCATION_CITY IN (
      'Miami','Miami Beach','Hialeah','Kendall','Doral','Homestead','Aventura','Coral Gables','North Miami','North Miami Beach','Cutler Bay','Miami Gardens','Pinecrest','Palmetto Bay','Sunny Isles Beach','Key Biscayne','Miami Lakes','Hialeah Gardens','Opa-locka',
      'Fort Lauderdale','Hollywood','Pompano Beach','Deerfield Beach','Coral Springs','Pembroke Pines','Miramar','Davie','Plantation','Sunrise','Hallandale Beach','Weston','Tamarac','Margate','Coconut Creek','Lauderhill','Dania Beach','Oakland Park','Wilton Manors','Parkland','Cooper City',
      'West Palm Beach','Boca Raton','Delray Beach','Boynton Beach','Palm Beach Gardens','Jupiter','Wellington','Palm Beach','Lake Worth','Royal Palm Beach','Riviera Beach','Greenacres','Lake Park','North Palm Beach')
)
SELECT
  COALESCE(NULLIF(TRIM(PRODUCT_CLASS), ''), '(blank)') AS boat_class,
  COUNT(*)                                             AS leads,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1)   AS pct,
  ROUND(MEDIAN(PRODUCT_PRICE), 0)                      AS median_price
FROM metro
GROUP BY 1
ORDER BY leads DESC
LIMIT 10;

This measures whether Miami buyers shop locally or reach far away. The boat location field stores the full word Florida, not the two letter code, so the local share compares against Florida. 


In [ ]:
WITH metro AS (
  SELECT LISTING_STATE, USER_LISTING_DISTANCE_IN_MILES AS dist
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND COALESCE(IS_SPAM_FINAL,0)=0
    AND USER_LOCATION_COUNTRY='USA'
    AND USER_LOCATION_REGION='Florida'
    AND USER_LOCATION_CITY IN (
      'Miami','Miami Beach','Hialeah','Kendall','Doral','Homestead','Aventura','Coral Gables','North Miami','North Miami Beach','Cutler Bay','Miami Gardens','Pinecrest','Palmetto Bay','Sunny Isles Beach','Key Biscayne','Miami Lakes','Hialeah Gardens','Opa-locka',
      'Fort Lauderdale','Hollywood','Pompano Beach','Deerfield Beach','Coral Springs','Pembroke Pines','Miramar','Davie','Plantation','Sunrise','Hallandale Beach','Weston','Tamarac','Margate','Coconut Creek','Lauderhill','Dania Beach','Oakland Park','Wilton Manors','Parkland','Cooper City',
      'West Palm Beach','Boca Raton','Delray Beach','Boynton Beach','Palm Beach Gardens','Jupiter','Wellington','Palm Beach','Lake Worth','Royal Palm Beach','Riviera Beach','Greenacres','Lake Park','North Palm Beach')
)
SELECT
  COUNT(*)                                                  AS metro_leads,
  ROUND(AVG(IFF(LISTING_STATE = 'Florida', 1, 0)) * 100, 1) AS pct_boat_in_florida,
  ROUND(MEDIAN(dist), 0)                                    AS median_distance_mi,
  ROUND(AVG(IFF(dist < 25, 1, 0)) * 100, 1)                 AS pct_under_25mi,
  ROUND(AVG(IFF(dist >= 500, 1, 0)) * 100, 1)               AS pct_500mi_plus
FROM metro;

This is the consumer level, running against the GA4 browsing event log. It counts browsing events, distinct device cookies, boat detail page views, and the distinct boats those detail pages covered, all for Miami metro browsers over twelve months. 

In [ ]:
SELECT
  COUNT(*)                              AS events,
  COUNT(DISTINCT DEVICE_ID)             AS distinct_devices,
  COUNT_IF(BDP_LISTING_ID IS NOT NULL)  AS bdp_views,
  COUNT(DISTINCT BDP_LISTING_ID)        AS distinct_boats_viewed
FROM AWS_DP_PROD.CONSUMER.GA4_CONSUMER_EVENTS_ENRICHED
WHERE EVENT_DATE >= DATEADD(month,-12,CURRENT_DATE)
  AND USER_LOCATION_REGION = 'Florida'
  AND USER_LOCATION_CITY IN (
    'Miami','Miami Beach','Hialeah','Kendall','Doral','Homestead','Aventura','Coral Gables','North Miami','North Miami Beach','Cutler Bay','Miami Gardens','Pinecrest','Palmetto Bay','Sunny Isles Beach','Key Biscayne','Miami Lakes','Hialeah Gardens','Opa-locka',
    'Fort Lauderdale','Hollywood','Pompano Beach','Deerfield Beach','Coral Springs','Pembroke Pines','Miramar','Davie','Plantation','Sunrise','Hallandale Beach','Weston','Tamarac','Margate','Coconut Creek','Lauderhill','Dania Beach','Oakland Park','Wilton Manors','Parkland','Cooper City',
    'West Palm Beach','Boca Raton','Delray Beach','Boynton Beach','Palm Beach Gardens','Jupiter','Wellington','Palm Beach','Lake Worth','Royal Palm Beach','Riviera Beach','Greenacres','Lake Park','North Palm Beach');

This is the customer level, showing which dealers receive Miami demand and how concentrated it is. It groups Miami dealer leads by the seller party, counts how many distinct dealers received any Miami lead, and reports the share of Miami leads held by the ten largest and the fifty largest dealers, plus the average leads per dealer. These dealers sit anywhere in the country, since a portion of the boats are outside Florida.

In [ ]:
WITH metro AS (
  SELECT PARTY_ID
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE)
    AND COALESCE(IS_SPAM_FINAL,0)=0
    AND NOT IS_FSBO
    AND PARTY_ID IS NOT NULL
    AND USER_LOCATION_COUNTRY='USA'
    AND USER_LOCATION_REGION='Florida'
    AND USER_LOCATION_CITY IN (
      'Miami','Miami Beach','Hialeah','Kendall','Doral','Homestead','Aventura','Coral Gables','North Miami','North Miami Beach','Cutler Bay','Miami Gardens','Pinecrest','Palmetto Bay','Sunny Isles Beach','Key Biscayne','Miami Lakes','Hialeah Gardens','Opa-locka',
      'Fort Lauderdale','Hollywood','Pompano Beach','Deerfield Beach','Coral Springs','Pembroke Pines','Miramar','Davie','Plantation','Sunrise','Hallandale Beach','Weston','Tamarac','Margate','Coconut Creek','Lauderhill','Dania Beach','Oakland Park','Wilton Manors','Parkland','Cooper City',
      'West Palm Beach','Boca Raton','Delray Beach','Boynton Beach','Palm Beach Gardens','Jupiter','Wellington','Palm Beach','Lake Worth','Royal Palm Beach','Riviera Beach','Greenacres','Lake Park','North Palm Beach')
),
agg AS (
  SELECT PARTY_ID, COUNT(*) AS leads
  FROM metro
  GROUP BY 1
),
ranked AS (
  SELECT PARTY_ID, leads, ROW_NUMBER() OVER (ORDER BY leads DESC) AS rnk
  FROM agg
)
SELECT
  COUNT(*)                                                     AS distinct_dealers,
  SUM(leads)                                                   AS metro_dealer_leads,
  ROUND(SUM(IFF(rnk <= 10, leads, 0)) * 100.0 / SUM(leads), 1) AS top10_dealer_share_pct,
  ROUND(SUM(IFF(rnk <= 50, leads, 0)) * 100.0 / SUM(leads), 1) AS top50_dealer_share_pct,
  ROUND(AVG(leads), 1)                                         AS avg_leads_per_dealer
FROM ranked;